# Recomendador por similitud coseno de género (sin modelo, sin entrenamiento)

Enfoque mucho más simple que `modelo_lightgbm.ipynb`: sin ML, solo estadística descriptiva + una fórmula.

1. **Score por libro**: promedio de calificaciones, solo para libros con **>= 3 calificaciones válidas**
   (menos que eso, no se puntúa — se descartó la alternativa de ponderar por cantidad de reviews).
2. **Similitud entre géneros**: para cada género, un vector de la calificación promedio que le dio cada
   lector (0 si no calificó nada de ese género). Similitud coseno entre esos vectores — dos géneros son
   "cercanos" si los mismos lectores los califican parecido (no si "suenan parecido" el nombre).
3. **Género principal de cada lector**: el género en el que dejó más calificaciones.
4. **Score efectivo** de un libro candidato para un lector = `similitud(género_libro, género_principal
   del lector) * score_promedio_del_libro`. Si coinciden, la similitud consigo mismo es 1, así que queda
   `1 * score_promedio_del_libro` — es el mismo caso general, no una excepción.
5. Para cada lector, top 20 libros no leídos por score efectivo descendente.

**Decisiones de diseño**:
- Reusamos el fuzzy matching de género de `dataset_features_genero.ipynb` (mismos 53 géneros
  canónicos) para no fragmentar la similitud por variantes de tipeo.
- Los libros sin género (`"desconocido"`, 61% del catálogo) quedan afuera de todo esto: no se les puede
  calcular una similitud de género con nada, así que no entran como candidatos ni como género principal.
- Para lectores sin ningún género principal definido (sin calificaciones, o todas en libros sin género)
  usamos un *fallback*: se ordenan los candidatos solo por `score_promedio_del_libro`, ignorando
  similitud de género — es la única forma sensata de darles una recomendación igual.
- La similitud es coseno sobre el rating promedio **crudo** (sin centrar por lector). Existe una
  variante más refinada, *similitud coseno ajustada* (restar primero el promedio de cada lector, para
  que no pese el sesgo de "este lector califica todo alto/bajo"), pero no la pediste y agrega una
  decisión extra — se puede sumar después si hace falta.

In [1]:
import re
import sqlite3
import unicodedata

import numpy as np
import pandas as pd
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

DB_PATH = "datos/data.db"
EJEMPLO_CSV = "datos/ejemplo.csv"
OUTPUT_CSV = "datos/recomendaciones_top20_similitud_genero.csv"
RANDOM_STATE = 42
MIN_RATINGS_LIBRO = 3  # libros con menos calificaciones no se puntuan


## 1. Carga y género canónico

Mismo criterio de limpieza que en los notebooks anteriores: se descarta la fila con fecha corrupta y
las interacciones sin metadata de libro o lector.

In [2]:
conn = sqlite3.connect(DB_PATH)
lectores = pd.read_sql_query("SELECT * FROM lectores", conn)
libros = pd.read_sql_query("SELECT * FROM libros", conn)
interacciones = pd.read_sql_query("SELECT * FROM interacciones", conn)
conn.close()

fecha_ok = interacciones["fecha"].str.match(r"^\d{2}-\d{2}-\d{4}$", na=False)
libro_ok = interacciones["id_libro"].isin(set(libros["id_libro"]))
lector_ok = interacciones["id_lector"].isin(set(lectores["id_lector"]))
validas = fecha_ok & libro_ok & lector_ok

inter = interacciones[validas].merge(libros[["id_libro", "genero"]], on="id_libro", how="left")
inter["rating"] = inter["rating"].astype(float)
print(f"interacciones validas: {len(inter):,} / {len(interacciones):,}")

interacciones validas: 461,073 / 461,408


In [3]:
def agrupar_por_similitud(items, threshold, scorer=fuzz.ratio):
    # une items en clusters (union-find) segun similitud; devuelve tambien los pares que dispararon cada union
    padre = {it: it for it in items}

    def find(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            padre[ra] = rb

    pares = []
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            score = scorer(items[i], items[j])
            if score >= threshold:
                pares.append((items[i], items[j], score))
                union(items[i], items[j])

    clusters = {}
    for it in items:
        clusters.setdefault(find(it), []).append(it)
    return clusters, pares


conteo_genero = inter.assign(g=inter["genero"].str.strip().str.lower()).dropna(subset=["g"])
conteo_genero = conteo_genero[conteo_genero["g"] != ""]["g"].value_counts().to_dict()
clusters_genero, _ = agrupar_por_similitud(sorted(conteo_genero.keys()), 80)

genero_canon = {}
for miembros in clusters_genero.values():
    canonico = max(miembros, key=lambda m: conteo_genero.get(m, 0))
    for m in miembros:
        genero_canon[m] = canonico

inter["genero_canon"] = inter["genero"].str.strip().str.lower().map(genero_canon)
print(f"generos canonicos: {inter['genero_canon'].nunique()}")
print(f"interacciones sin genero (excluidas de todo lo que sigue): {inter['genero_canon'].isna().sum():,} "
      f"({inter['genero_canon'].isna().mean():.0%})")

inter_g = inter.dropna(subset=["genero_canon"]).copy()

generos canonicos: 52
interacciones sin genero (excluidas de todo lo que sigue): 1 (0%)


## 2. Score promedio por libro (>= 3 calificaciones)

In [4]:
libro_stats = inter_g.groupby("id_libro").agg(
    genero_libro=("genero_canon", "first"), avg_score=("rating", "mean"), n_ratings=("rating", "size")
)
candidatos = libro_stats[libro_stats["n_ratings"] >= MIN_RATINGS_LIBRO].copy()
print(f"libros con genero: {len(libro_stats):,}  candidatos (>= {MIN_RATINGS_LIBRO} calificaciones): {len(candidatos):,}")
candidatos.head()

libros con genero: 48,062  candidatos (>= 3 calificaciones): 18,706


,genero_libro,avg_score,n_ratings
id_libro,,,
10-anos-con-mafalda,humor,8.539474,76
100-balas-colgando-de-un-hilo-1,"cómics, novela gráfica",8.200000,5
100-cien-poemas,"poesía, teatro",8.250000,4
100-vistas-de-tokio,"cómics, novela gráfica",9.000000,4
1000-sitios-que-ver-en-espana-al-menos-una-vez-en-la-vida,lecturas complementarias,6.000000,5


## 3. Matriz lector × género y similitud coseno entre géneros

In [5]:
matriz_lector_genero = (
    inter_g.groupby(["id_lector", "genero_canon"])["rating"].mean().unstack(fill_value=0.0)
)
print(f"matriz lector x genero: {matriz_lector_genero.shape}")

generos = matriz_lector_genero.columns.tolist()
sim = cosine_similarity(matriz_lector_genero.T.values)
similitud_genero = pd.DataFrame(sim, index=generos, columns=generos)

print(f"diagonal (debe ser 1.0): min={np.diag(sim).min():.6f} max={np.diag(sim).max():.6f}")
similitud_genero.round(2).iloc[:6, :6]

matriz lector x genero: (10667, 52)
diagonal (debe ser 1.0): min=1.000000 max=1.000000


,arte,autoayuda y espiritualidad,"biografías, memorias",clásicos de la literatura,cocina,"cómics, novela gráfica"
arte,1.00,0.03,0.11,0.08,0.02,0.11
autoayuda y espiritualidad,0.03,1.00,0.19,0.17,0.09,0.13
"biografías, memorias",0.11,0.19,1.00,0.58,0.11,0.40
clásicos de la literatura,0.08,0.17,0.58,1.00,0.09,0.43
cocina,0.02,0.09,0.11,0.09,1.00,0.09
"cómics, novela gráfica",0.11,0.13,0.40,0.43,0.09,1.00


## 4. Género principal por lector

In [6]:
conteo_lector_genero = inter_g.groupby(["id_lector", "genero_canon"]).size()
genero_principal = conteo_lector_genero.groupby("id_lector").idxmax().apply(lambda t: t[1])
print(f"lectores con genero principal definido: {len(genero_principal):,}")
genero_principal.head()

lectores con genero principal definido: 10,667


id_lector
-2             novela negra, intriga, terror
00sally00        fantástica, ciencia ficción
01826084e      novela negra, intriga, terror
02jjulia       novela negra, intriga, terror
040213cesar    novela negra, intriga, terror
dtype: str

## 5. Ranking de candidatos por género principal (evita el cross-join completo)

El score efectivo de un libro candidato solo depende del género principal del lector (no del lector en
sí) — así que en vez de cruzar cada lector contra todos los candidatos, precalculamos un ranking
completo de candidatos **por cada uno de los géneros principales posibles**, y para cada lector filtramos
los ya leídos de su ranking correspondiente. Mucho más liviano que el cross-join batched que usamos con
LightGBM, y no hace falta procesar por lotes.

**Desempate**: con el promedio simple y el piso de 3 calificaciones, es común que varios libros de nicho
empaten justo en el score máximo. Ante un empate en `score_efectivo`, ordenamos por `n_ratings`
descendente — a igual score, preferimos el libro con más calificaciones detrás (más confiable).

In [7]:
ranking_por_genero = {}
for g in generos:
    sim_g = candidatos["genero_libro"].map(similitud_genero[g])
    score_efectivo = sim_g * candidatos["avg_score"]
    ranking_por_genero[g] = (
        candidatos.assign(score_efectivo=score_efectivo)
        .sort_values(["score_efectivo", "n_ratings"], ascending=[False, False])
        [["genero_libro", "score_efectivo", "n_ratings"]]
    )

ranking_fallback = (
    candidatos.assign(score_efectivo=candidatos["avg_score"])
    .sort_values(["score_efectivo", "n_ratings"], ascending=[False, False])
    [["genero_libro", "score_efectivo", "n_ratings"]]
)
print(f"{len(ranking_por_genero)} rankings precalculados (uno por genero principal) + 1 fallback")

52 rankings precalculados (uno por genero principal) + 1 fallback


## 6. Top 20 por lector para `ejemplo.csv`

In [8]:
ejemplo = pd.read_csv(EJEMPLO_CSV)
lectores_objetivo = sorted(ejemplo["id_lector"].unique())
print(f"lectores objetivo: {len(lectores_objetivo)}")

leidos_por_lector = interacciones.groupby("id_lector")["id_libro"].apply(set).to_dict()

sin_genero_principal = sum(1 for l in lectores_objetivo if l not in genero_principal.index)
print(f"lectores objetivo sin genero principal (fallback a ranking global): {sin_genero_principal}")

filas = []
for lid in lectores_objetivo:
    g_principal = genero_principal.get(lid)
    ranking = ranking_por_genero[g_principal] if g_principal in ranking_por_genero else ranking_fallback
    leidos = leidos_por_lector.get(lid, set())
    top20 = ranking[~ranking.index.isin(leidos)].head(20)
    for id_libro, fila in top20.iterrows():
        filas.append((lid, id_libro, fila["score_efectivo"], fila["n_ratings"]))

recomendaciones = pd.DataFrame(filas, columns=["id_lector", "id_libro", "score_efectivo", "n_ratings"])
print(f"total: {recomendaciones.shape}")

lectores objetivo: 832


lectores objetivo sin genero principal (fallback a ranking global): 17


total: (16640, 4)


## 7. Validaciones de sanidad

In [9]:
filas_por_lector = recomendaciones.groupby("id_lector").size()
print("lectores objetivo cubiertos:", filas_por_lector.index.isin(lectores_objetivo).all() and len(filas_por_lector) == len(lectores_objetivo))
print(filas_por_lector.value_counts())

incompletos = filas_por_lector[filas_por_lector < 20]
if len(incompletos):
    print(f"\n{len(incompletos)} lectores con menos de 20 recomendaciones:")
    print(incompletos)

dup = recomendaciones.duplicated(subset=["id_lector", "id_libro"]).sum()
ya_leidos_set = {(lid, lib) for lid, libs in leidos_por_lector.items() for lib in libs}
recomendados_set = set(map(tuple, recomendaciones[["id_lector", "id_libro"]].itertuples(index=False, name=None)))
interseccion = recomendados_set & ya_leidos_set

print(f"\npares duplicados: {dup}")
print(f"recomendaciones que ya estaban leidas (deberia ser 0): {len(interseccion)}")

assert dup == 0, "hay pares (lector, libro) duplicados"
assert len(interseccion) == 0, "se recomendo un libro ya leido"
print("\nOK: sin duplicados, sin libros ya leidos.")

lectores objetivo cubiertos: True
20    832
Name: count, dtype: int64

pares duplicados: 0
recomendaciones que ya estaban leidas (deberia ser 0): 0

OK: sin duplicados, sin libros ya leidos.


## 8. CSV final (misma estructura que `ejemplo.csv`)

In [10]:
salida = (
    recomendaciones.sort_values(["id_lector", "score_efectivo", "n_ratings"], ascending=[True, False, False])
    [["id_lector", "id_libro"]]
    .reset_index(drop=True)
)

print("columnas:", salida.columns.tolist(), "== ejemplo.csv:", salida.columns.tolist() == ejemplo.columns.tolist())
print("filas:", len(salida), " (ejemplo.csv tiene", len(ejemplo), ")")

salida.to_csv(OUTPUT_CSV, index=False)
print(f"\nguardado en {OUTPUT_CSV}")
salida.head(10)

columnas: ['id_lector', 'id_libro'] == ejemplo.csv: True
filas: 16640  (ejemplo.csv tiene 16640 )

guardado en datos/recomendaciones_top20_similitud_genero.csv


,id_lector,id_libro
0,05-03-1970,novelas-2
1,05-03-1970,el-sargento-cadaver
2,05-03-1970,la-espera-1
3,05-03-1970,la-flor-contada
4,05-03-1970,la-llave-del-espejo
5,05-03-1970,las-desventuras-de-jonas-plum
6,05-03-1970,el-tercer-encuentro
7,05-03-1970,climas
8,05-03-1970,infortunio
9,05-03-1970,judas


## 9. Ejemplo

In [11]:
muestra = recomendaciones["id_lector"].drop_duplicates().sample(2, random_state=RANDOM_STATE).tolist()
for lid in muestra:
    g_principal = genero_principal.get(lid, "(sin genero principal -> fallback)")
    top5 = recomendaciones[recomendaciones["id_lector"] == lid].sort_values("score_efectivo", ascending=False).head(5)
    top5 = top5.merge(libros[["id_libro", "titulo", "autor", "genero"]], on="id_libro", how="left")
    print(f"\n=== top 5 para {lid} (genero principal: {g_principal}) ===")
    print(top5[["id_libro", "titulo", "autor", "genero", "score_efectivo", "n_ratings"]].to_string(index=False))


=== top 5 para omallorqui (genero principal: novela negra, intriga, terror) ===
                                      id_libro                                          titulo                         autor                        genero  score_efectivo  n_ratings
el-animal-mas-peligroso-un-thriller-victoriano EL ANIMAL MÁS PELIGROSO. Un thriller victoriano        POMBO, GABRIEL ANTONIO Novela negra, intriga, terror       10.000000          3
                   el-dulce-amargor-del-crimen                     EL DULCE AMARGOR DEL CRIMEN               NOIRÉ, PHILIPPE Novela negra, intriga, terror       10.000000          3
                                el-nino-pajaro                                  EL NIÑO PÁJARO PEÑATE RODRÍGUEZ, JUAN MANUEL Novela negra, intriga, terror        9.800000          5
           matanza-de-atocha-1977-caso-abierto           MATANZA DE ATOCHA, 1977: CASO ABIERTO           GALLO, ALEJANDRO M. Novela negra, intriga, terror        9.666667          3
         

**Resumen**: sin modelo ni entrenamiento — score por libro (promedio, >= 3 calificaciones) × similitud
coseno entre el género del libro y el género principal del lector (calculada sobre el perfil de rating
promedio por lector×género). Libros y lectores sin género quedan fuera (o con fallback al ranking
global, para que ningún lector de `ejemplo.csv` se quede sin recomendaciones). Resultado en
`datos/recomendaciones_top20_similitud_genero.csv`, misma estructura que `ejemplo.csv`.